# Phase 3: Data Preparation & Train/Test Design

**Business question this phase serves:** *how do we split and encode the data so that every model
we fit from Phase 4 onward is evaluated fairly: with no leakage, and with a train/validation/test
split that is demonstrably not biased?*

**Input:** [`data/processed/policy_level_clean.parquet`](../data/processed/policy_level_clean.parquet),
built in [Phase 1](01_data_acquisition_and_quality_checks.ipynb). Every encoding decision below is
justified by a specific finding from [Phase 2](02_exploratory_data_analysis.ipynb): this notebook
does not introduce a new decision without pointing back to the evidence for it.

**What this notebook does:**
1. Split the portfolio into train / validation / test, **stratified on claim occurrence** (claims are
   rare, ~5% of policies, so an unstratified split risks unlucky over/under-representation of the
   very rows the frequency model cares about most).
2. **Prove the split is not biased**, the exact requirement of this phase, by comparing
   exposure-weighted frequency, severity, and category mix across the three sets side by side.
3. Build a **feature encoder** (`src/features/pipeline.py`) that is fit once on the training set only
   and applied identically everywhere else, and show directly that it does not leak information from
   validation/test back into how features are built.
4. Persist train/validation/test splits and the fitted encoder, ready for Phase 4 (frequency GLM).

**A design note on style:** the encoder in `src/features/pipeline.py` is written as plain, explicit
Python rather than an `sklearn.Pipeline`/`ColumnTransformer`: a few more lines of code in exchange for
every step (what gets capped, at what threshold, learned from which data) being visible and readable
instead of hidden inside a framework object, a trade-off chosen for code that should be easy to
read and check.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def find_project_root(marker: str = "requirements.txt") -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (looked for '{marker}' upward from {here})")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.pipeline import (
    PolicyFeatureEncoder,
    compare_split_balance,
    get_exposure_offset,
    get_frequency_target,
    get_severity_subset,
    train_val_test_split_policies,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "policy_level_clean.parquet")
print(f"Loaded {len(df):,} policies.")

Loaded 668,626 policies.


## 1. Train / validation / test split

Split ratio: **60% train / 20% validation / 20% test**, which leaves about 133,700 policies in each of
the validation and test sets: enough for stable evaluation metrics on a rare event (only about 3.7% of
policies have a claim), while keeping most of the data for fitting.

**Why stratify on claim count:** claims are rare, so a plain random split can put noticeably more or
fewer of the claiming policies in one set than another. Stratifying on a claim-count bucket (`0`, `1`,
`2+`) keeps that proportion the same across the three sets, checked right below.

In [2]:
train_df, val_df, test_df = train_val_test_split_policies(df, val_size=0.20, test_size=0.20, random_state=42)

print(f"Train      : {len(train_df):>7,} policies ({len(train_df) / len(df):.1%})")
print(f"Validation : {len(val_df):>7,} policies ({len(val_df) / len(df):.1%})")
print(f"Test       : {len(test_df):>7,} policies ({len(test_df) / len(df):.1%})")
print(f"\nRow counts sum back to original: {len(train_df) + len(val_df) + len(test_df) == len(df)}")

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    claim_rate = (part["ClaimNb_freq_target"] > 0).mean()
    print(f"  {name:<12} share of policies with >=1 claim: {claim_rate:.4%}")

Train      : 401,175 policies (60.0%)
Validation : 133,725 policies (20.0%)
Test       : 133,726 policies (20.0%)

Row counts sum back to original: True
  train        share of policies with >=1 claim: 3.6902%
  validation   share of policies with >=1 claim: 3.6897%
  test         share of policies with >=1 claim: 3.6904%


**Finding:** the split lands almost exactly on the target 60/20/20 ratio, and the share of
policies with at least one claim is essentially identical across all three sets (matching to four
decimal places): exactly what correct stratification should produce.

## 2. Bias check: do train / validation / test look like samples of the *same* portfolio?

This is the check the user of this pipeline should never skip: stratifying on claim count guarantees
the *frequency* target's rate matches across splits, but says nothing about severity, or about whether
one split accidentally got an unusual mix of, say, `VehBrand`s. Both are checked directly below.

In [3]:
balance = compare_split_balance(train_df, val_df, test_df)
balance

,split,n_policies,share_of_total,exposure_weighted_frequency,avg_severity
0,train,401175,0.6000,0.0740,"2,005.6315"
1,validation,133725,0.2000,0.0740,"3,341.8886"
2,test,133726,0.2000,0.0741,"1,975.5157"


**Reading:** exposure-weighted frequency is the same across the three sets (7.40-7.41%, matching
the Phase 2 portfolio figure), as expected from the stratified split. Average severity differs
(train about EUR 2,006, validation about EUR 3,342, test about EUR 1,976). The next cell checks whether
that is a bias.

In [4]:
# Severity is heavy-tailed (Phase 1/2: a single claim in this dataset reaches EUR 4.08M against a
# portfolio mean of ~EUR 2,300). With claims split across only ~15,000/~5,000/~5,000 policies in
# train/val/test, exactly *which* set a handful of extreme claims lands in can move the *mean* by a
# large amount, purely by chance, while the *typical* (median) claim, and the count of large claims
# relative to each set's size, tells a much steadier story.
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    sev = get_severity_subset(part)
    n_large = (sev["AvgClaimAmount"] > 100_000).sum()
    print(
        f"{name:<12} n_paid_claims={len(sev):>6,}  median_severity=EUR {sev['AvgClaimAmount'].median():>7,.0f}  "
        f"mean_severity=EUR {sev['AvgClaimAmount'].mean():>8,.0f}  claims>EUR100k={n_large:>2}  max=EUR {sev['AvgClaimAmount'].max():>10,.0f}"
    )

train        n_paid_claims=14,792  median_severity=EUR   1,172  mean_severity=EUR    1,962  claims>EUR100k=22  max=EUR    774,412
validation   n_paid_claims= 4,928  median_severity=EUR   1,172  mean_severity=EUR    3,293  claims>EUR100k=12  max=EUR  4,075,401
test         n_paid_claims= 4,931  median_severity=EUR   1,172  mean_severity=EUR    1,936  claims>EUR100k= 5  max=EUR    276,643


**Finding: the split is not biased.** The **median** paid claim is EUR 1,172 in every split. (This
exact amount is one of a few standard settlement values that recur thousands of times in
`freMTPL2sev`, which is why the medians coincide exactly.) The mean gap comes from where a small number
of extreme claims landed: the validation set holds the largest claim in the dataset (EUR 4.08M) and 12
claims above EUR 100k, against 22 in the three-times-larger training set and 5 in the test set. The
mean moves, the median does not: the expected behavior of a random split of heavy-tailed data, and the
reason Phase 6 treats large losses separately.

In [5]:
# A quick categorical-mix check too, for completeness: category shares should be close across splits.
for col in ["VehGas", "VehBrand"]:
    mix = pd.concat(
        [
            train_df[col].value_counts(normalize=True).rename("train"),
            val_df[col].value_counts(normalize=True).rename("validation"),
            test_df[col].value_counts(normalize=True).rename("test"),
        ],
        axis=1,
    )
    print(f"\n--- {col} share by split ---")
    print(mix)


--- VehGas share by split ---
         train  validation   test
VehGas                           
Regular 0.5089      0.5068 0.5061
Diesel  0.4911      0.4932 0.4939

--- VehBrand share by split ---
          train  validation   test
VehBrand                          
B12      0.2412      0.2421 0.2422
B1       0.2411      0.2397 0.2410
B2       0.2361      0.2380 0.2363
B3       0.0794      0.0794 0.0788
B5       0.0513      0.0518 0.0515
B6       0.0424      0.0424 0.0423
B4       0.0376      0.0363 0.0376
B10      0.0263      0.0263 0.0266
B11      0.0202      0.0205 0.0200
B13      0.0183      0.0174 0.0178
B14      0.0060      0.0062 0.0058


**Finding:** category shares for `VehGas` and `VehBrand` (representative of the nominal factors)
match across splits to within a fraction of a percentage point, as expected from a large, simple random
sample of a stable population. The split passes every bias check applied to it.

## 3. Feature encoding: fit on train only

Recap of the encoding rule (`src/features/pipeline.py`, decisions justified in Phase 2):

| Column | Treatment | Why |
|---|---|---|
| `DrivAge`, `BonusMalus`, `VehPower` | pass through unchanged | no data-quality issue found; shape of the effect (linear/binned/smooth) is a Phase 4/5 modeling choice |
| `Density` | `log(Density)` | spans 4 orders of magnitude (Phase 1: 1 to 27,000): log scale is far more stable for a GLM/GAM |
| `VehAge` | capped at the 99.5th percentile **learned from train** | Phase 1 found values up to 100 years, not a plausible vehicle age |
| `VehBrand`, `VehGas`, `Region` | one-hot, `drop="first"`, unseen category -> all zeros | standard GLM-safe categorical encoding; safe against a brand-new category appearing later at scoring time |
| `Area` | **dropped** | Phase 2 Section 4: empirically redundant with `Density` (medians span 27 to 27,000 with zero overlap across the 6 codes) |
| `Exposure` | **not a feature**: returned separately as `log(Exposure)` | the GLM/GAM offset, not a rating factor (see `get_exposure_offset`) |

In [6]:
encoder = PolicyFeatureEncoder()
encoder.fit(train_df)   # <-- the only call to .fit() in this whole notebook, and it only ever sees train_df

print(f"VehAge capping threshold learned from TRAIN ONLY (99.5th percentile): {encoder.vehage_cap_:.1f} years")
print(f"Raw VehAge max in train : {train_df['VehAge'].max()}")
print(f"Raw VehAge max in val   : {val_df['VehAge'].max()}  (encoder has never seen this value)")
print(f"Raw VehAge max in test  : {test_df['VehAge'].max()}")

X_train = encoder.transform(train_df)
X_val = encoder.transform(val_df)
X_test = encoder.transform(test_df)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val   shape: {X_val.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"\nFeature columns ({len(encoder.feature_names_)}):")
print(encoder.feature_names_)

VehAge capping threshold learned from TRAIN ONLY (99.5th percentile): 24.0 years
Raw VehAge max in train : 100
Raw VehAge max in val   : 100  (encoder has never seen this value)
Raw VehAge max in test  : 100



X_train shape: (401175, 37)
X_val   shape: (133725, 37)
X_test  shape: (133726, 37)

Feature columns (37):
['DrivAge', 'BonusMalus', 'VehPower', 'log_Density', 'VehAge_capped', 'VehBrand_B10', 'VehBrand_B11', 'VehBrand_B12', 'VehBrand_B13', 'VehBrand_B14', 'VehBrand_B2', 'VehBrand_B3', 'VehBrand_B4', 'VehBrand_B5', 'VehBrand_B6', 'VehGas_Regular', 'Region_R21', 'Region_R22', 'Region_R23', 'Region_R24', 'Region_R25', 'Region_R26', 'Region_R31', 'Region_R41', 'Region_R42', 'Region_R43', 'Region_R52', 'Region_R53', 'Region_R54', 'Region_R72', 'Region_R73', 'Region_R74', 'Region_R82', 'Region_R83', 'Region_R91', 'Region_R93', 'Region_R94']


In [7]:
# Direct proof that transform() on val/test uses TRAIN's threshold, not one recomputed from their own data.
print(f"VehAge_capped max in X_train : {X_train['VehAge_capped'].max()}")
print(f"VehAge_capped max in X_val   : {X_val['VehAge_capped'].max()}   <- capped at train's threshold, even though raw val VehAge reaches {val_df['VehAge'].max()}")
print(f"VehAge_capped max in X_test  : {X_test['VehAge_capped'].max()}")

assert X_train["VehAge_capped"].max() <= encoder.vehage_cap_
assert X_val["VehAge_capped"].max() <= encoder.vehage_cap_
assert X_test["VehAge_capped"].max() <= encoder.vehage_cap_
print("\nAll three sets respect the train-only cap. No leakage.")

for name, X in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    assert X.isnull().sum().sum() == 0, f"{name} has missing values after encoding!"
print("No missing values in any encoded feature matrix.")

VehAge_capped max in X_train : 24
VehAge_capped max in X_val   : 24   <- capped at train's threshold, even though raw val VehAge reaches 100
VehAge_capped max in X_test  : 24

All three sets respect the train-only cap. No leakage.
No missing values in any encoded feature matrix.


**Finding:** the cap learned from `train_df` (99.5th percentile) is applied identically to
validation and test: both of which contain raw `VehAge` values far above that threshold (up to 100
years), and both come back capped at exactly the training threshold, never a value recomputed from
their own data. That is the concrete, checkable meaning of "no leakage" for this pipeline.

## 4. Targets, offset, and the severity subset

Three more pieces every model in Phase 4-6 will need, each kept explicit and separate rather than
bundled into one object: see `src/features/pipeline.py` docstrings for why each one is not a feature.

In [8]:
y_freq_train = get_frequency_target(train_df)
offset_train = get_exposure_offset(train_df)

print("Frequency target y (ClaimNb): first 5:", y_freq_train.head().tolist())
print(f"Offset = log(Exposure), range: [{offset_train.min():.3f}, {offset_train.max():.3f}]")
print(f"(offset of 0.0 = full year of exposure; more negative = shorter exposure)")

sev_train = get_severity_subset(train_df)
sev_val = get_severity_subset(val_df)
sev_test = get_severity_subset(test_df)

X_sev_train = encoder.transform(sev_train)  # exact same fitted encoder: frequency and severity share one feature-building rule
print(f"\nSeverity training subset: {len(sev_train):,} policies with >=1 paid claim (of {len(train_df):,} total)")
print(f"X_sev_train shape: {X_sev_train.shape}")

Frequency target y (ClaimNb): first 5: [1, 0, 0, 0, 0]
Offset = log(Exposure), range: [-5.903, 0.000]
(offset of 0.0 = full year of exposure; more negative = shorter exposure)

Severity training subset: 14,792 policies with >=1 paid claim (of 401,175 total)
X_sev_train shape: (14792, 37)


## 5. Persist the split and the fitted encoder

Saved once here, reused by every notebook from Phase 4 onward: the same discipline as Phase 1/2:
one source of truth, never recomputed differently in two places.

In [9]:
processed_dir = PROJECT_ROOT / "data" / "processed"
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(processed_dir / "train.parquet", index=False)
val_df.to_parquet(processed_dir / "val.parquet", index=False)
test_df.to_parquet(processed_dir / "test.parquet", index=False)

joblib.dump(encoder, models_dir / "feature_encoder.joblib")

print("Saved:")
print(f"  {(processed_dir / 'train.parquet').relative_to(PROJECT_ROOT).as_posix()}  ({len(train_df):,} rows)")
print(f"  {(processed_dir / 'val.parquet').relative_to(PROJECT_ROOT).as_posix()}    ({len(val_df):,} rows)")
print(f"  {(processed_dir / 'test.parquet').relative_to(PROJECT_ROOT).as_posix()}   ({len(test_df):,} rows)")
print(f"  {(models_dir / 'feature_encoder.joblib').relative_to(PROJECT_ROOT).as_posix()}  (fit on train only, {len(encoder.feature_names_)} output features)")

Saved:
  data/processed/train.parquet  (401,175 rows)
  data/processed/val.parquet    (133,725 rows)
  data/processed/test.parquet   (133,726 rows)
  models/feature_encoder.joblib  (fit on train only, 37 output features)


## Phase 3 summary

| Check | Result |
|---|---|
| Split ratio | 60% / 20% / 20% (train / validation / test) |
| Claim-rate stratification | Identical to 4 decimal places across the three sets |
| Exposure-weighted frequency across splits | 7.40% / 7.40% / 7.41% |
| Median severity across splits | EUR 1,172 in all three (the mean gap comes from a few very large claims) |
| Categorical mix (`VehGas`, `VehBrand`) across splits | Matches within a fraction of a percentage point |
| Encoder leakage check | `VehAge` cap learned from train only; applied identically to val/test raw values up to 100 years |
| Missing values after encoding | None, in any of the three sets |

**Output:**
[`data/processed/train.parquet`](../data/processed/train.parquet),
[`val.parquet`](../data/processed/val.parquet),
[`test.parquet`](../data/processed/test.parquet), and
[`models/feature_encoder.joblib`](../models/feature_encoder.joblib), ready for
**Phase 4: frequency model (Poisson GLM)**.